# Suv oylari — v3 (fenologiya)

`ekin_yerlar_qisqa.gpkg` dan **`ekin_yerlar_qisqa_v3_fenologiya.gpkg`** yasaladi. Asl faylga tegilmaydi.

O'zgaradigan ustunlar — **faqat**:
- `suv_boshi`, `suv_oxiri` — barcha mavsumli konturlarda;
- `season` — faqat Ozuqa yoki Bug'doy ustidagi Paxta konturlarida (`Mavsumiy ekin` → `Ikkilamchi ekin`).

Geometriya va boshqa atributlar o'zgarmaydi (fayl nusxalanadi, nusxada SQL UPDATE; 6-qadam buni tekshiradi).

Qoidalar — `config/suv_oylari.yaml`:
1. **Birlamchi** — hammasi 3-oydan, tugashi ekin turi bo'yicha (3-5 yoki 3-6; Paxta ostidagi Ozuqa/Bug'doy 3-4).
2. **Ikkilamchi** — shu daladagi Birlamchi tugagan oydan keyin, 10-oygacha (3-5 → 6-10, 3-6 → 7-10).
3. **Mavsumiy** — 3-10.
4. **Ozuqa / Bug'doy ustidagi Paxta** — Ikkilamchi ekin, 5-10.

Bitta dala: ikki kontur kesishmasi ulardan birining maydonining ≥ 50% i.

**Muhit:** kernel — `fenologiya` conda muhiti. Birinchi marta (bir marta) PowerShell da:
```
& "$env:LOCALAPPDATA\miniconda3\Scripts\conda.exe" install -n fenologiya -c conda-forge ipykernel -y
```
so'ng VS Code da *Select Kernel → Python Environments → fenologiya*.

In [ ]:
# 0. Sozlash — fayl yo'llari shu yerda
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "src"))
import fenologiya.muhit  # noqa: F401  (DLL/GDAL — numpy dan oldin)
import pandas as pd
from fenologiya import suv

KIRISH = suv.standart_kirish()            # bir_ikki_lamchi/natija/ekin_yerlar_qisqa.gpkg
VERSIYA = "v3"
CHIQISH_PAPKA = os.path.dirname(KIRISH)    # natija asl fayl yonida
NOM = f"{os.path.splitext(os.path.basename(KIRISH))[0]}_{VERSIYA}_fenologiya"
CHIQISH = os.path.join(CHIQISH_PAPKA, NOM + ".gpkg")
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200)
print("Kirish :", KIRISH); print("Chiqish:", CHIQISH)

## 1. Qoidalar (config/suv_oylari.yaml)

In [ ]:
cfg, zm = suv.sozlamalar()
print(open(os.path.join(ROOT, "config", "suv_oylari.yaml"), encoding="utf-8").read())

## 2. Konturlarni o'qish

In [ ]:
df, mc = suv.oqi(KIRISH)               # atributlar
G = suv.geom_oqi(KIRISH, mc)           # mavsumli konturlar geometriyasi
print(f"{len(df)} kontur, mavsum ustuni: {mc}")
for x in suv.nomlarni_tekshir(df, cfg, zm):
    print("OGOHLANTIRISH:", x)
df.groupby(["mavsum", "suv_boshi", "suv_oxiri"], dropna=False).size().rename("kontur (hozir)")

## 3. Hisob: Paxta → Ikkilamchi va suv oylari

In [ ]:
df, L = suv.hisobla(df, G, cfg, zm)
d = df[df.mavsum.notna()].assign(yangi=lambda x: [suv.oy_matn(b, o) for b, o in zip(x.b, x.o)])
(d.groupby(["yangi_mavsum", "yangi"]).agg(kontur=("uniqueid", "size"), maydon_ga=("maydon", "sum")).round().astype(int))

### 3a. Ikkilamchiga o'tgan Paxta konturlari (viloyat / tuman)

In [ ]:
px = df[df.mavsum != df.yangi_mavsum]
print(f"{len(px)} kontur, {px.maydon.sum():,.0f} ga")
px.groupby(["viloyat", "tuman"]).agg(kontur=("uniqueid", "size"), maydon_ga=("maydon", "sum")).round().sort_values("kontur", ascending=False)

## 4. O'zgarish jadvali (eski → yangi) — CSV ga

In [ ]:
T = suv.ozgarish_jadvali(df)
out_csv = os.path.join(CHIQISH_PAPKA, NOM + "_ozgarish.csv")
T.to_csv(out_csv, index=False, encoding="utf-8-sig"); print("->", out_csv)
T[T.maydon_ga >= 500]

## 5. GPKG yozish (~1 daqiqa, ~400 MB nusxa)

In [ ]:
h = suv.gpkg_yangila(KIRISH, CHIQISH, df, mc)
print("->", CHIQISH); print("yangilangan qatorlar:", h)

## 6. Tekshiruv: suv / season dan boshqa hech narsa o'zgarmaganmi (geometriya ham)

In [ ]:
farq = suv.faqat_suv_farqi(KIRISH, CHIQISH)
print("suv_boshi/suv_oxiri/season dan boshqa farq qilgan qatorlar:", farq)
assert all(v == 0 for v in farq.values()), "DIQQAT: boshqa ustunlar o'zgargan!"
v3, _ = suv.oqi(CHIQISH)
v3.groupby(["mavsum", "suv_boshi", "suv_oxiri"], dropna=False).size().rename("kontur (v3)")

## 7. Birlamchi va Ikkilamchi konturlar chegarasi — bir xil / farqli

v3 fayl bo'yicha (Ikkilamchiga o'tgan Paxta ham kiradi). Har kontur uchun qarama-qarshi mavsumdagi eng mos juft:
- **aynan bir xil** — polygon chegarasi to'liq teng;
- **deyarli bir xil** — IoU ≥ 0.99 (farq < 1%, chegarada mayda tafovut);
- **farq qiladi** — bitta dalada (kesishma ≥ 50%), lekin chegara boshqa (IoU < 0.99);
- **jufti yo'q** — qarama-qarshi mavsumda ≥ 50% kesishadigan kontur yo'q.

In [ ]:
G3 = suv.geom_oqi(CHIQISH, mc)
C = suv.chegara_solishtir(G3)
pd.crosstab(C.tomon, C.toifa, margins=True, margins_name="Jami")

In [ ]:
# viloyat bo'yicha
pd.crosstab([C.tomon, C.viloyat], C.toifa, margins=True, margins_name="Jami")

In [ ]:
# bitta konturga nechta juft to'g'ri keladi (1 dan ko'p — dala bo'lingan)
C.groupby("tomon").juft_soni.value_counts().unstack(fill_value=0)

In [ ]:
# farq qiladigan va jufti yo'q konturlar ro'yxati — CSV (ArcGIS da uniqueid bo'yicha topish uchun)
out = os.path.join(CHIQISH_PAPKA, NOM + "_chegara_farqli.csv")
C[C.toifa.isin(["farq qiladi", "jufti yo'q"])].to_csv(out, index=False, encoding="utf-8-sig")
print("->", out)
C[C.toifa == "farq qiladi"].groupby("tomon").iou.describe().round(2)